# Protein Context and Druggability with Pharos

The NIH Common Fund Illuminating the Druggable Genome, or IDG, program develops knowledge and tools for understudied proteins in druggable families. Pharos combines target information from many sources. Review protein knowledge and target development without treating druggability as proof of disease causality.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
pharos = pd.read_csv(DATA_DIR / "pharos_target_context.csv")
pharos

## Optional live GraphQL request

The function shows the current query. The saved table remains the default input so the analysis can be repeated.

In [ ]:
PHAROS_GRAPHQL_URL = "https://pharos-api.ncats.io/graphql"
TARGET_QUERY = """
query TargetContext($symbol: String!) {
  target(q: {sym: $symbol}) {
    sym
    name
    uniprot
    tdl
    publicationCount
    ligandCounts { name value }
    ppiCounts { name value }
  }
}
"""


def fetch_pharos_target(
    gene_symbol: str,
    timeout_seconds: int = 30,
) -> dict[str, object]:
    """Return current Pharos context for one gene symbol."""
    response = requests.post(
        PHAROS_GRAPHQL_URL,
        json={
            "query": TARGET_QUERY,
            "variables": {"symbol": gene_symbol},
        },
        timeout=timeout_seconds,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("errors"):
        raise RuntimeError(payload["errors"])
    return payload["data"]["target"]


# Example, intentionally not executed during routine notebook runs:
# current_myh7 = fetch_pharos_target("MYH7")

In [ ]:
target_summary = pharos.loc[
    :,
    [
        "gene_symbol",
        "target_name",
        "tdl",
        "drug_count",
        "publication_count",
        "ppi_count",
    ],
].sort_values(["tdl", "gene_symbol"])
target_summary

## Interpretation

`MYH7` is Tclin in this saved result, while the other candidates are Tbio. This difference describes target development. It does not rank the variants by pathogenicity or the genes by disease importance.